# The Price is Right: Full ML Pipeline

A product **price predictor** that goes from naive baselines all the way to frontier LLMs,
measuring Mean Absolute Error (MAE) at each step.

Data comes from the HuggingFace dataset `ed-donner/items_lite`.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from openai import OpenAI

from datasets import load_dataset
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
print(f"OpenRouter API Key: {openrouter_api_key[:4]}...")

openai_client = OpenAI(
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1"
)

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print("All imports OK")

## Load the dataset from HuggingFace

In [ ]:
# Load the lite dataset (faster iteration during development)
# For the full 800K+ item dataset use "ed-donner/items_full"
ds = load_dataset("ed-donner/items_lite")
print(ds)

train_data = list(ds["train"])
val_data   = list(ds["validation"])
test_data  = list(ds["test"])

print(f"\nTrain: {len(train_data):,} | Val: {len(val_data):,} | Test: {len(test_data):,}")
print("\nSample item:")
print(train_data[0])

In [ ]:
# Extract prices and descriptions
def get_texts(data):  return [item["summary"] for item in data]
def get_prices(data): return [item["price"] for item in data]

train_texts  = get_texts(train_data)
train_prices = get_prices(train_data)
val_texts    = get_texts(val_data)
val_prices   = get_prices(val_data)
test_texts   = get_texts(test_data)
test_prices  = get_prices(test_data)

# Quick stats
prices = train_prices + val_prices + test_prices
print(f"Price range: ${min(prices):.0f} – ${max(prices):.0f}")
print(f"Mean: ${np.mean(prices):.2f} | Median: ${np.median(prices):.2f}")

## Baseline models

In [ ]:
results = {}  # model_name → test_MAE

# Baseline 1: predict the mean price
mean_price = np.mean(train_prices)
preds_mean = [mean_price] * len(test_prices)
results["Constant (mean)"] = mean_absolute_error(test_prices, preds_mean)

# Baseline 2: random price within training range
lo, hi = min(train_prices), max(train_prices)
preds_rand = [random.uniform(lo, hi) for _ in test_prices]
results["Random"] = mean_absolute_error(test_prices, preds_rand)

for name, mae in results.items():
    print(f"{name:<30} MAE = ${mae:.2f}")

## Classical ML (bag-of-words features)

In [ ]:
# CountVectorizer + Linear Regression
lr_pipeline = Pipeline([
    ("vect", CountVectorizer(max_features=2000, stop_words="english")),
    ("reg",  Ridge()),
])
lr_pipeline.fit(train_texts, train_prices)
preds_lr = lr_pipeline.predict(test_texts)
preds_lr = np.clip(preds_lr, 1, 1000)
results["Ridge Regression (BoW)"] = mean_absolute_error(test_prices, preds_lr)

# Random Forest
rf_pipeline = Pipeline([
    ("vect", CountVectorizer(max_features=500, stop_words="english")),
    ("reg",  RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)),
])
rf_pipeline.fit(train_texts, train_prices)
preds_rf = rf_pipeline.predict(test_texts)
results["Random Forest (BoW)"] = mean_absolute_error(test_prices, preds_rf)

for name, mae in results.items():
    print(f"{name:<35} MAE = ${mae:.2f}")

## Deep Learning (PyTorch MLP on TF-IDF features)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

# Build TF-IDF matrix
tfidf = TfidfVectorizer(max_features=1000, stop_words="english")
X_train = tfidf.fit_transform(train_texts).toarray().astype(np.float32)
X_val   = tfidf.transform(val_texts).toarray().astype(np.float32)
X_test  = tfidf.transform(test_texts).toarray().astype(np.float32)

y_train = np.array(train_prices, dtype=np.float32)
y_val   = np.array(val_prices,   dtype=np.float32)
y_test  = np.array(test_prices,  dtype=np.float32)

# PyTorch MLP
class PriceMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512,    256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256,    128), nn.ReLU(),
            nn.Linear(128,      1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = PriceMLP(X_train.shape[1]).to(device)
opt    = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.HuberLoss()

train_dl = DataLoader(
    TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
    batch_size=128, shuffle=True
)

# Train for 15 epochs
for epoch in range(15):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss_fn(model(xb), yb).backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        val_preds = model(torch.tensor(X_val).to(device)).cpu().numpy()
    val_mae = mean_absolute_error(y_val, np.clip(val_preds, 1, 1000))
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}  val MAE = ${val_mae:.2f}")

model.eval()
with torch.no_grad():
    test_preds_nn = model(torch.tensor(X_test).to(device)).cpu().numpy()
results["PyTorch MLP"] = mean_absolute_error(y_test, np.clip(test_preds_nn, 1, 1000))
print(f"\nNeural Net test MAE = ${results['PyTorch MLP']:.2f}")

## Frontier LLM (GPT-4o-mini zero-shot)

We test on a small subset to keep API costs minimal.

In [ ]:
from tqdm import tqdm

PREFIX = "Price is $"
QUESTION = "What does this cost to the nearest dollar?"
LLM_MODEL = "gpt-4o-mini"
N_LLM_TEST = 25   # keep API costs low; increase for better estimates


def llm_predict_price(description: str) -> float:
    """Ask GPT to predict the price of the product."""
    prompt = (
        f"{QUESTION}\n\n{description}\n\n"
        "Reply with ONLY a number (the price to the nearest dollar, no $ sign)."
    )
    response = openai_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    try:
        return float(response.choices[0].message.content.strip().replace(",", ""))
    except ValueError:
        return float(mean_price)


# Run on a small subset
subset_texts  = test_texts[:N_LLM_TEST]
subset_prices = test_prices[:N_LLM_TEST]

llm_preds = []
for text in tqdm(subset_texts, desc=f"GPT-4o-mini zero-shot (n={N_LLM_TEST})"):
    llm_preds.append(llm_predict_price(text))

results[f"GPT-4o-mini zero-shot (n={N_LLM_TEST})"] = mean_absolute_error(subset_prices, llm_preds)
print(f"GPT-4o-mini zero-shot MAE = ${results[f'GPT-4o-mini zero-shot (n={N_LLM_TEST})']:.2f}")

## Results Summary & Visualisation

In [ ]:
print("\n" + "="*55)
print(f"{'Model':<40} {'MAE':>10}")
print("-"*55)
for name, mae in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{name:<40} ${mae:>8.2f}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results.keys())
maes  = list(results.values())
bars  = ax.barh(names, maes, color="steelblue")
ax.bar_label(bars, fmt="$%.0f", padding=4)
ax.set_xlabel("Mean Absolute Error ($)")
ax.set_title("Price Prediction MAE — lower is better")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("price_prediction_results.png", dpi=120)
plt.show()
print("\nChart saved to price_prediction_results.png")